# Prompt Optimization with DSPy- Let an Algorithm Write the Prompt

**Module 5 · Lesson 5.6**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Why optimize prompts - compile, don't hand-tune
- Measure first - LLM-as-judge and a test set
- DSPy - declare what, not how
- Optimize - let the optimizer search
- Compare and ship safely - A/B, regression, canary
- The landscape and the project

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## The whole game: from "feels better" to "is measurably better"

> ⚙️ **Analogy**
>
> **A hand-written prompt is assembly code you tune by hand.** You nudge a word, re-read the output, and guess whether it improved. It works, but it does not scale, and you can never be sure a different phrasing would not have been better.
> **DSPy is a compiler.** You declare *what* you want (the task) and a *metric* for "good", and it generates and tunes the actual prompt for you - the way a compiler turns intent into optimized machine code. You stop writing prompt strings and start writing programs and metrics.
> **Where the analogy breaks down:** a real compiler is deterministic and provably correct; a prompt optimizer is a stochastic search against a fallible metric. Point it at a biased judge or a tiny dev set and it will happily "compile" a *worse* prompt. The optimizer is only as good as the metric - and the metric is yours to get right.

Every technique here is tested with **real API calls** and paired with a plain-English walkthrough. We use Gemini through the current unified SDK, and DSPy on top of it; the ideas transfer to any model DSPy can drive.

- **Construct** an evaluation harness - a metric and a held-out test set, including an LLM-as-judge - to score a prompt objectively.

- **Explain** why prompts are programs that can be compiled, and how DSPy separates the task (signature) from the wording (the optimized prompt).

- **Apply** a DSPy optimizer (BootstrapFewShot or MIPROv2) to auto-tune instructions and few-shot demonstrations against a metric.

- **Evaluate** optimization results honestly - guarding against dev-set overfitting and LLM-as-judge bias - and choose an optimizer for the task.

## Warm-up: 60 seconds of recall

Tap each card to check yourself against earlier lessons. Each one is load-bearing for today.

## The setup: helpers we will reuse all lesson

Examples call this `generate` on the current unified **google-genai** SDK (the older `google.generativeai` was deprecated in 2025 - [migration guide](https://ai.google.dev/gemini-api/docs/migrate)), and configure **DSPy** to drive the same model.

**📝 `setup.py Gemini +`** - *DSPy*

In [ ]:
# pip install "google-genai>=1.0.0" dspy>=2.5
from google import genai
from google.genai import types
import os, dspy

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-3-flash"

def generate(prompt: str) -> str:
    try:
        return client.models.generate_content(model=MODEL, contents=prompt).text.strip()
    except Exception as e:
        return f"[API error: {e}]"

# Point DSPy at the same model - it builds and tunes prompts through this.
dspy.configure(lm=dspy.LM(f"gemini/{MODEL}", api_key=os.environ["GOOGLE_API_KEY"]))
print("ready")
# Output: ready

- `generate` is our plain-Gemini call (used for the LLM-as-judge metric in Step 2) on the unified `genai.Client` - the deprecated package used a global `configure()` plus a per-call `GenerativeModel`.

- `dspy.configure(lm=dspy.LM("gemini/gemini-3-flash", api_key=...))` tells DSPy which model to compile prompts for - one line, and DSPy drives that model under the hood. Pass `api_key` explicitly: DSPy routes through LiteLLM, whose `gemini/` provider looks for `GEMINI_API_KEY`, so hand it your `GOOGLE_API_KEY` (or export both).

- DSPy never replaces the model; it generates and tunes the *prompt and few-shot demonstrations* you send to it.

- The `try/except` keeps a single network blip from crashing an optimization run that can make many calls.

---

## 🎯 Concept 1: Why optimize prompts - compile, don't hand-tune

### Why optimize prompts - compile, don't hand-tune

Hand-tuning wording does not scale and gives you no proof. Compiling does both.

**Tuning a radio by ear vs by a signal meter.** By ear you stop when it "sounds clear enough"; with a meter you turn the dial to the actual peak and know you are there.

The gain: a metric is the signal meter for prompts. Once you can measure quality, you can let a search find the peak instead of guessing.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened? Hand-tuning is effort without a metric, so it wanders and plateaus; optimization is a measured objective plus a search, so it climbs and converges. **The reframe of this lesson is that a prompt is a program you compile against a metric** - and the rest follows: first you need the metric (Step 2), then the program (Step 3), then the search (Step 4).

---

## 🎯 Concept 2: Measure first - LLM-as-judge and a test set

### Measure first - LLM-as-judge and a test set

You cannot optimize what you cannot measure. Build the metric before the optimizer.

**📝 `metric.py Gemini`** - *API*

```python
def judge(question: str, answer: str, gold: str) -> float:
    """LLM-as-judge: score 0-1 against an explicit rubric (anchored, not vibes)."""
    prompt = f"""Score the ANSWER from 0 to 1 using ONLY this rubric:
- correct vs the reference (most weight)
- grounded, not vague
- concise (do NOT reward length)

Question: {question}
Reference: {gold}
Answer: {answer}
Return ONLY a number between 0 and 1."""
    try:
        return float(generate(prompt))
    except ValueError:
        return 0.0

def evaluate(predict, testset) -> float:
    """Average the judge score over a HELD-OUT test set."""
    scores = [judge(x["q"], predict(x["q"]), x["gold"]) for x in testset]
    return sum(scores) / len(scores)

print(round(evaluate(my_prompt_fn, test_set), 2))
# Output: 0.71      (a number you can now improve - and defend)
```

- **The rubric is explicit.** The judge scores against named criteria (correctness, grounding, conciseness) and is told *not* to reward length - an anchor against verbosity bias.

- **The reference matters.** Passing a `gold` answer turns a vague "is this good?" into "does it match the reference?", which judges do far more reliably.

- **It runs on a held-out test set.** `evaluate` averages over data the optimizer never trains on - the only number you should trust or report.

- This is the measure-it discipline from Lesson 5.1's reliability harness, now turned into the objective an optimizer will maximize.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened? An LLM-as-judge turns free-form outputs into a number you can optimize, but the rubric *is* the objective - so a biased judge becomes a biased objective. Anchor it to explicit criteria and a human-checked gold set, and always score on a held-out test set. **The metric you build now is exactly what the optimizer will chase** - get it right before you run anything.

---

## 🎯 Concept 3: DSPy - declare what, not how

### DSPy - declare what, not how

Describe the task as a typed signature; let DSPy own the actual prompt wording.

**📝 `dspy_signature.py`** - *DSPy*

In [ ]:
import dspy

class Classify(dspy.Signature):
    """Classify a customer support ticket into one category."""
    ticket: str = dspy.InputField()
    category: str = dspy.OutputField(desc="Hardware, Software, Connectivity, Billing, or Account")

classify = dspy.ChainOfThought(Classify)      # module = HOW it reasons

result = classify(ticket="My laptop won't connect to the company VPN")
print(result.category)
# Output: Connectivity         (DSPy wrote the prompt; you wrote the SIGNATURE)

- **The signature is the *what*.** `Classify` declares a typed input (`ticket`) and output (`category`) and a one-line docstring - the same "declare the shape" instinct as the Pydantic schemas in Lesson 5.5.

- **The module is the *how*.** `dspy.ChainOfThought(Classify)` wraps the signature in a reasoning strategy (here, think-then-answer); swapping to `dspy.Predict` or `dspy.ReAct` changes the strategy without touching the task.

- **You never wrote the prompt string.** DSPy assembles the actual instruction and format from the signature - and the optimizer (Step 4) will rewrite it against your metric.

- This separation is the whole point: the task is stable, the wording is generated, and the optimizer owns the wording.

#### 💡 What just happened

⚡ What just happened? You described the task as a typed signature and picked a reasoning module - and DSPy produced the working prompt. **You declared *what*, not *how the words go***, which is exactly what lets an optimizer rewrite the wording later without you re-engineering anything. This is prompting as programming.

---

## 🎯 Concept 4: Optimize - let the optimizer search

### Optimize - let the optimizer search

An optimizer proposes prompts and few-shot sets, scores them, and keeps the best.

**📝 `optimize.py`** - *DSPy*

```python
from dspy.teleprompt import MIPROv2

def metric(example, pred, trace=None) -> float:
    return 1.0 if pred.category == example.category else 0.0   # your objective (gold-label match)

def evaluate_acc(program, dataset) -> float:
    """Mean of the SAME metric over a split - our accuracy there."""
    return sum(metric(ex, program(**ex.inputs())) for ex in dataset) / len(dataset)

# MIPROv2 jointly tunes the INSTRUCTIONS and the FEW-SHOT DEMOS.
optimizer = MIPROv2(metric=metric, auto="light")              # "light" = small budget
compiled = optimizer.compile(classify, trainset=train, valset=dev)

# Always score the result on a HELD-OUT test set, not dev.
before = evaluate_acc(classify, test)
after  = evaluate_acc(compiled, test)
print(f"hand-written: {before:.0%}   compiled: {after:.0%}")
# Output: hand-written: 71%   compiled: 88%   (held-out test, not dev)
```

- **The metric is the objective.** This task has *gold labels*, so `metric` scores exact-match on the category and `evaluate_acc` reuses it - simpler than Step 2's LLM-as-judge, which you reach for only when there is no single right answer to match against. Either way the optimizer maximizes exactly this, so it must reward what you actually want.

- **MIPROv2 searches two things at once** - the instruction wording and the few-shot demonstration set - proposing candidates, scoring them on the validation set, and keeping the best (BootstrapFewShot just searches demos; GEPA adds natural-language-feedback evolutionary search and is the current state of the art).

- **`compile` returns a tuned program,** not a string - the same `classify` module with better instructions and demos baked in.

- **Three splits, three jobs.** `trainset` is where the optimizer draws candidate few-shot demos; `valset` (dev) is what it scores each candidate against to pick the best; `test` is the untouched set you report. Score on `test`, never `dev` - a dev number flatters the optimizer because it chose against dev.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

A team runs an optimizer until the metric on their training/dev split looks great, declares victory, and ships. In production it scores far lower - worse than the hand-written prompt it replaced. The optimizer had *memorized the dev set* (overfitting), not learned the task. The fix is non-negotiable: a held-out test set the optimizer never sees, reported separately, and stop optimizing when the test number stops rising.

#### 💡 What just happened

⚡ What just happened? A DSPy optimizer searches over instructions and few-shot demos, scoring candidates against your metric and keeping the best - lifting the hand-written baseline well past where hand-tuning plateaued on the held-out test, with no prompt strings written by hand. **The optimizer is a search; your metric and your held-out set are the guardrails** that keep it honest. GEPA is the 2026 state of the art; MIPROv2 and BootstrapFewShot are the workhorses.

---

## 🎯 Concept 5: Compare and ship safely - A/B, regression, canary

### Compare and ship safely - A/B, regression, canary

A compiled prompt is a candidate. Prove it beats the baseline before it goes live.

**📝 `compare_and_ship.py`** - *Production*

```python
# 1. A/B: score both on the SAME held-out test set
ab = {"baseline": evaluate(baseline_fn, test),
      "compiled": evaluate(compiled_fn, test)}

# 2. Regression: never lose on cases you already got right
def no_regressions(new_fn, golden_cases) -> bool:
    return all(new_fn(c["q"]) == c["expected"] for c in golden_cases)

# 3. Canary: send a small % of live traffic to the new prompt first
def route(query, traffic_pct=5, user_bucket=0):
    return compiled_fn(query) if user_bucket < traffic_pct else baseline_fn(query)

print(ab, "| regressions ok:", no_regressions(compiled_fn, golden))
# Output: {'baseline': 0.71, 'compiled': 0.88} | regressions ok: True
```

- **A/B on the same test set.** Scoring both prompts on identical held-out data is the only fair comparison - the compiled prompt must win cleanly, not by a noise-sized margin.

- **Regression suite.** `no_regressions` guards a set of cases you already handle correctly, so an "improvement" that breaks a known-good case is caught before launch.

- **Canary rollout.** `route` sends a small slice of traffic to the new prompt first; if live metrics hold, you widen it - the same safe-deploy discipline as code.

- This is prompt CI/CD; the full LLMOps version (versioning, monitoring, automated eval gates) is Module 14.

#### 💡 What just happened

⚡ What just happened? A compiled prompt is a candidate, not a guarantee: you A/B it against the baseline on the same held-out set, run a regression suite so no known-good case breaks, and canary it on a slice of live traffic before the full rollout. **Prompts are code, so you ship them like code** - measured, gated, and reversible.

---

## 🎯 Concept 6: The landscape and the project

### The landscape and the project

The eval tools, the optimizer choices, and the full pipeline end to end.

> 📦 **Info**
>
> The optimizer and eval landscape (2026)
> - **BootstrapFewShot** - bootstraps good few-shot demonstrations from your data. Cheap; start here.
> - **MIPROv2** - jointly optimizes instructions and demonstrations. The reliable default for most tasks.
> - **GEPA** - genetic-Pareto search with natural-language feedback: it mutates and recombines candidate prompts like an evolutionary search, and instead of only a numeric score it feeds the model *written critiques* of its own mistakes to steer the next candidate. The 2026 state of the art, reported to exceed MIPROv2 by a clear margin at a fraction of the rollouts. Use for hard tasks.
> - **Eval tools** - promptfoo, RAGAS, DeepEval supply the metrics and test harnesses optimization runs against.

**📝 `pipeline.py Complete`** - *project*

```python
# The whole lesson, end to end: declare -> optimize -> evaluate honestly -> A/B.
from dspy.teleprompt import MIPROv2

program = dspy.ChainOfThought(Classify)              # Step 3: declare the task
optimizer = MIPROv2(metric=metric, auto="light")        # Step 4: search
compiled = optimizer.compile(program, trainset=train, valset=dev)

baseline_acc = evaluate_acc(program, test)           # Step 2 metric, held-out test
compiled_acc = evaluate_acc(compiled, test)

if compiled_acc > baseline_acc and no_regressions(compiled, golden):
    print(f"Ship via canary: {baseline_acc:.0%} -> {compiled_acc:.0%}")   # Step 5
else:
    print("Keep the baseline - no honest improvement.")
# Output: Ship via canary: 71% -> 88%
```

- **Declare (Step 3)** the task as a DSPy program - no prompt string by hand.

- **Optimize (Step 4)** with MIPROv2 against your metric on train/dev.

- **Evaluate honestly (Step 2)** - both programs on the held-out `test` set, never dev.

- **Gate the ship (Step 5)** - only ship (via canary) if it beats the baseline AND passes the regression suite; otherwise keep the baseline.

- The full eval-driven-development loop - error analysis, automated eval gates, monitoring - is Module 14; this is its prompt-level core.

#### 💡 What just happened

⚡ What just happened? The pipeline runs the whole discipline in order: declare, optimize, evaluate on a held-out set, and ship only on an honest, regression-checked win. Eval tools supply the metrics; the optimizer ladder (BootstrapFewShot -> MIPROv2 -> GEPA) supplies the search. **This is prompt engineering grown up - measured, compiled, and gated** - and it hands straight off to eval-driven development in Module 14.

## Putting it together: prompt engineering, compiled

| Stage | What you do | The guardrail |
|---|---|---|
| Measure (Step 2) | Build an LLM-as-judge metric + test set | Anchor the rubric; held-out set |
| Declare (Step 3) | Write a DSPy signature + module | Task is stable; wording is generated |
| Optimize (Step 4) | Run BootstrapFewShot / MIPROv2 / GEPA | Score on dev, report on test |
| Ship (Step 5) | A/B, regression suite, canary | Reversible, gated rollout |

### 🧮 The whole lesson on one screen

**Compile, don't hand-tune** - a prompt is a program you optimize against a metric (Step 1). **Measure first** - an anchored LLM-as-judge on a held-out set, because the metric is the objective (Step 2). **Declare what, not how** - a DSPy signature owns the task, DSPy owns the wording (Step 3). **Let the optimizer search** instructions and demos, and report on the held-out test, not dev (Step 4). **Ship like code** - A/B, regression, canary (Step 5). And **pick the optimizer** for the task, from BootstrapFewShot to GEPA (Step 6). Above all: you cannot optimize what you cannot measure, and you cannot trust what you did not hold out.

Forward hooks you just planted: this is the prompt-level core of eval-driven development - we'll build the full evaluation and error-analysis discipline in Module 14 (LLMOps), and we'll come back to the cost of optimization runs in Module 13.

- Khattab et al., *DSPy: Compiling Declarative Language Model Calls* (2023) - [arxiv.org/abs/2310.03714](https://arxiv.org/abs/2310.03714); Opsahl-Ong et al., *MIPRO* (2024) - [arxiv.org/abs/2406.11695](https://arxiv.org/abs/2406.11695)

- Zhou et al., *APE: LLMs Are Human-Level Prompt Engineers* (2022) - [arxiv.org/abs/2211.01910](https://arxiv.org/abs/2211.01910); Yang et al., *OPRO: LLMs as Optimizers* (2023) - [arxiv.org/abs/2309.03409](https://arxiv.org/abs/2309.03409)

- Pryzant et al., *Automatic Prompt Optimization with Gradient Descent* (ProTeGi, 2023) - [arxiv.org/abs/2305.03495](https://arxiv.org/abs/2305.03495)

- Zheng et al., *Judging LLM-as-a-Judge* (2023) - [arxiv.org/abs/2306.05685](https://arxiv.org/abs/2306.05685); LangProBe (2025) - [arxiv.org/abs/2502.20315](https://arxiv.org/abs/2502.20315)

- DSPy docs and optimizer guide - [dspy.ai](https://dspy.ai); eval tools [promptfoo](https://www.promptfoo.dev), [RAGAS](https://docs.ragas.io), [DeepEval](https://docs.confident-ai.com)

- Google google-genai migration guide - [ai.google.dev/gemini-api/docs/migrate](https://ai.google.dev/gemini-api/docs/migrate)

## Key takeaways

> ℹ️ **Info**
>
> What you learned - 6 ideas
> - **Compile, don't hand-tune** - a prompt is a program optimized against a metric; hand-tuning wanders and plateaus.
> - **Measure first** - an anchored LLM-as-judge on a held-out test set; the metric is the objective, so beware judge bias.
> - **Declare what, not how** - DSPy signatures own the task; modules own the reasoning; DSPy owns the wording.
> - **Let the optimizer search** - BootstrapFewShot, MIPROv2, GEPA tune instructions and demos; report on held-out test.
> - **Ship like code** - A/B against the baseline, run a regression suite, canary the rollout.
> - **Pick the optimizer** - cheap BootstrapFewShot to state-of-the-art GEPA; eval tools supply the metrics.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Prompt Optimization with DSPy- Let an Algorithm Write the Prompt**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-5_6.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-5.6.html` - regenerate this notebook whenever the source HTML is updated.*
